# correction-memory: one corrective demo as a gated activation memory in SmolVLA (LIBERO)
**Branch:** `feature/correction-memory` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all. Clones, installs lerobot 0.7.0 + LIBERO, caches HF assets on Drive, runs the
GPU pytest pre-flight (aborts if invalid), runs the resumable campaign (results on Drive), prints RESULTS.md, unassigns the GPU.
Re-running after a disconnect resumes from Drive.

In [ ]:
# Step 1: Mount Drive first: results AND the HF cache (checkpoint, SmolVLM2, lerobot/libero demos) live there
import os  # first cell: the Drive prompt appears right away
from google.colab import drive
drive.mount("/content/drive")
base = "/content/drive/MyDrive/moonshot_results"
os.environ["CM_RESULTS"] = f"{base}/correction-memory"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.makedirs(os.environ["CM_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["CM_RESULTS"], "| existing:", sorted(os.listdir(os.environ["CM_RESULTS"])))

In [ ]:
# Step 2: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 3: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | vCPUs: {os.cpu_count()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/correction-memory"
target_dir = "/content/correction-memory"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 4: Installs. lerobot 0.7.0 with LIBERO (hf-libero, MuJoCo) and SmolVLA extras. hf-libero asks interactively for a
# dataset path on first import -> answer "n" once to write ~/.libero/config.yaml with the default paths.
subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgl1", "libosmesa6"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)  # hf-egl-probe builds with CMake (no py3.13 wheel)
# egl_probe / hf-egl-probe (deps of hf-libero/robomimic) fail to build on Colab's Python 3.13. They only pick an EGL GPU;
# install same-name stubs (v1.0.2, get_available_devices() -> [0]) so pip skips the build. Rendering uses MUJOCO_GL=egl.
for _dist, _mod in (("hf-egl-probe", True), ("egl_probe", False)):
    _d = f"/content/egl_stubs/{_dist}"; os.makedirs(_d, exist_ok=True)
    open(f"{_d}/pyproject.toml", "w").write(
        '[build-system]\nrequires = ["setuptools"]\nbuild-backend = "setuptools.build_meta"\n'
        f'[project]\nname = "{_dist}"\nversion = "1.0.2"\n[tool.setuptools]\npy-modules = {["egl_probe"] if _mod else []}\n'.replace("'", '"'))
    if _mod:
        open(f"{_d}/egl_probe.py", "w").write("def get_available_devices():\n    return [0]\n")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", _d], check=True)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], capture_output=True, text=True)
if _r.returncode:  # Colab hides child-process output: print pip's real error before failing
    print(_r.stdout[-4000:]); print(_r.stderr[-6000:])
_r.check_returncode()
print(subprocess.run([sys.executable, "-c", "import libero.libero"], input="n\n", capture_output=True, text=True).stdout[-500:])
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print(subprocess.run([sys.executable, "-c", "import lerobot, transformers, torch; print('lerobot', lerobot.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 5: Pre-flight on the GPU (real checkpoint, real demos, one LIBERO env): no-op hooks bit-identical, own-chunk
# teacher-forced shift = 0, injection at the captured site, gating, GN sharp dirs, LoRA restores weights, every LIBERO task
# maps to demos, policy chunk closer to its demo than to a shuffled one, init-state control. ABORTS (and unassigns) on failure.
env = {**os.environ, "CM_REAL": "1"}
_t = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_cm.py"], env=env, capture_output=True, text=True)  # Colab hides child output
print(_t.stdout[-20000:]); print(_t.stderr[-5000:])
os.makedirs("/content/drive/MyDrive/moonshot_results/correction-memory", exist_ok=True); open("/content/drive/MyDrive/moonshot_results/correction-memory/pytest_log.txt", "w").write(_t.stdout + _t.stderr)
if _t.returncode:
    raise RuntimeError("pytest pre-flight failed on GPU -> campaign aborted (INVALID)")
print("[Pre-flight] all checks passed on GPU.")

In [ ]:
# Step 6: Full campaign (resumable): pilot -> selection -> memory write -> dev (kill rule) -> test -> LoRA -> secondary -> gate
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Running rollout" not in line and "Loading weights" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["CM_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()